<a href="https://colab.research.google.com/github/JCF-the-creator/Multi-Motorsport-Analytics---Desempenho-e-Estrat-gia/blob/main/F1_World_Analistyc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importar biblioteca**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

**Denifir os caminhos**

In [15]:
RAW_PATH = Path('/content/')
SILVER_PATH = Path('data/silver')
GOLD_PATH = Path('data/gold')

**Carregar todos os CSVs**

In [16]:
datasets = {
    'circuits': pd.read_csv(RAW_PATH / 'circuits.csv'),
    'constructor_results': pd.read_csv(RAW_PATH / 'constructor_results.csv'),
    'constructor_standings': pd.read_csv(RAW_PATH / 'constructor_standings.csv'),
    'constructors': pd.read_csv(RAW_PATH / 'constructors.csv'),
    'drivers': pd.read_csv(RAW_PATH / 'drivers.csv'),
    'driver_standings': pd.read_csv(RAW_PATH / 'driver_standings.csv'),
    'lap_times': pd.read_csv(RAW_PATH / 'lap_times.csv'),
    'pit_stops': pd.read_csv(RAW_PATH / 'pit_stops.csv'),
    'qualifying': pd.read_csv(RAW_PATH / 'qualifying.csv'),
    'races': pd.read_csv(RAW_PATH / 'races.csv'),
    'results': pd.read_csv(RAW_PATH / 'results.csv'),
    'seasons': pd.read_csv(RAW_PATH / 'seasons.csv'),
    'sprint_results': pd.read_csv(RAW_PATH / 'sprint_results.csv'),
    'status': pd.read_csv(RAW_PATH / 'status.csv')
   }

**Criando os datasets** | **Verificando duplicatas e Nulo**

In [30]:
for name, df in datasets.items():
  print(name, df.shape)

circuits (77, 9)
constructor_results (12625, 5)
constructor_standings (13391, 7)
constructors (212, 5)
drivers (861, 9)
driver_standings (34863, 7)
lap_times (589081, 6)
pit_stops (11371, 7)
qualifying (10494, 9)
races (1125, 18)
results (26759, 18)
seasons (75, 2)
sprint_results (360, 16)
status (139, 2)


In [10]:
for name, df in datasets.items():
  print(name, df.duplicated().sum())

circuits 0
constructor_results 0
constructor_standings 0
consturctors 0
driver_standings 0
lap_times 0
pit_stops 0
qualifying 0
races 0
results 0
seasons 0
sprint_results 0
status 0


Criando dimensões dos pilotos

In [41]:
drivers = datasets["drivers"].copy()

drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]

dim_driver = drivers[
    [
        "driverId",
        "driverRef",
        "driver_name",
        "dob",
        "nationality"
    ]
]

**Criando Dimensões das equipes**

In [19]:
constructors = datasets["constructors"].copy()

dim_constructor = constructors[
    [
        "constructorId",
        "constructorRef",
        "name",
        "nationality"
    ]
].rename(columns={
    "name": "team_name"
})

**Criando dimensões de circuitos**

In [20]:
circuits = datasets["circuits"].copy()

dim_circuit = circuits[
    [
        "circuitId",
        "circuitRef",
        "name",
        "location",
        "country",
        "lat",
        "lng",
        "alt"
    ]
].rename(columns={
    "name": "circuit_name"
})

**Criando dimensão de corridas**

In [21]:
races = datasets["races"].copy()

dim_race = races[
    [
        "raceId",
        "year",
        "round",
        "circuitId",
        "name",
        "date"
    ]
].rename(columns={
    "name": "race_name"
})

**Criando tabela fato de resultados**

In [22]:
results = datasets["results"].copy()

results["positionOrder"] = pd.to_numeric(
    results["positionOrder"],
    errors="coerce"
)

results["grid"] = pd.to_numeric(
    results["grid"],
    errors="coerce"
)

results["points"] = pd.to_numeric(
    results["points"],
    errors="coerce"
)

fato_resultados = results.copy()

fato_resultados["is_winner"] = fato_resultados["positionOrder"] == 1
fato_resultados["is_podium"] = fato_resultados["positionOrder"] <= 3
fato_resultados["is_top10"] = fato_resultados["positionOrder"] <= 10
fato_resultados["started_from_pole"] = fato_resultados["grid"] == 1
fato_resultados["position_gain"] = fato_resultados["grid"] - fato_resultados["positionOrder"]

**Criando base principla Gold**

In [23]:
f1_gold = fato_resultados.merge(
    dim_driver,
    on="driverId",
    how="left"
)

f1_gold = f1_gold.merge(
    dim_constructor,
    on="constructorId",
    how="left"
)

f1_gold = f1_gold.merge(
    dim_race,
    on="raceId",
    how="left"
)

f1_gold = f1_gold.merge(
    dim_circuit,
    on="circuitId",
    how="left"
)

In [24]:
wins_by_driver = (
    f1_gold[f1_gold["is_winner"]]
    .groupby("driver_name")
    .size()
    .reset_index(name="wins")
    .sort_values("wins", ascending=False)
)

In [25]:
podiums_by_driver = (
    f1_gold[f1_gold["is_podium"]]
    .groupby("driver_name")
    .size()
    .reset_index(name="podiums")
    .sort_values("podiums", ascending=False)
)

In [26]:
points_by_team = (
    f1_gold
    .groupby("team_name")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

In [29]:
filtered_f1_gold = f1_gold[f1_gold["started_from_pole"]]

pole_conversion_data = {
    "total_poles": filtered_f1_gold["started_from_pole"].sum(),
    "wins_from_pole": filtered_f1_gold["is_winner"].sum()
}
pole_conversion = pd.Series(pole_conversion_data)

pole_conversion["conversion_rate"] = (
    pole_conversion["wins_from_pole"] / pole_conversion["total_poles"]
)

**Criando análise de pit stops**

In [31]:
pit_stops = datasets["pit_stops"].copy()

pit_stops["pit_seconds"] = pit_stops["milliseconds"] / 1000

pit_gold = pit_stops.merge(
    f1_gold[
        [
            "raceId",
            "driverId",
            "driver_name",
            "team_name",
            "year",
            "race_name"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

In [32]:
avg_pit_by_team = (
    pit_gold
    .groupby("team_name")["pit_seconds"]
    .mean()
    .reset_index()
    .sort_values("pit_seconds")
)

In [33]:
pit_count_by_team = (
    pit_gold
    .groupby("team_name")["stop"]
    .count()
    .reset_index(name="pit_stop_count")
    .sort_values("pit_stop_count", ascending=False)
)

**Criando analise de tempos de volta**

In [34]:
lap_times = datasets["lap_times"].copy()

lap_times["lap_seconds"] = lap_times["milliseconds"] / 1000

lap_gold = lap_times.merge(
    f1_gold[
        [
            "raceId",
            "driverId",
            "driver_name",
            "team_name",
            "year",
            "race_name"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

In [35]:
driver_consistency = (
    lap_gold
    .groupby("driver_name")["lap_seconds"]
    .agg(
        avg_lap_time="mean",
        best_lap_time="min",
        lap_consistency="std"
    )
    .reset_index()
    .sort_values("lap_consistency")
)

Criando analise de classifição

In [36]:
qualifying = datasets["qualifying"].copy()

qualifying_gold = qualifying.merge(
    f1_gold[
        [
            "raceId",
            "driverId",
            "driver_name",
            "team_name",
            "year",
            "race_name",
            "positionOrder",
            "is_winner",
            "is_podium"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

In [37]:
qualifying_gold["qualified_top3"] = qualifying_gold["position"] <= 3

qualifying_impact = (
    qualifying_gold
    .groupby("qualified_top3")
    .agg(
        corridas=("raceId", "count"),
        vitorias=("is_winner", "sum"),
        podios=("is_podium", "sum")
    )
    .reset_index()
)

qualifying_impact["taxa_vitoria"] = (
    qualifying_impact["vitorias"] / qualifying_impact["corridas"]
)

qualifying_impact["taxa_podio"] = (
    qualifying_impact["podios"] / qualifying_impact["corridas"]
)

**Criando analise de sprint**

In [38]:
sprint = datasets["sprint_results"].copy()

sprint_gold = sprint.merge(
    dim_driver,
    on="driverId",
    how="left"
).merge(
    dim_constructor,
    on="constructorId",
    how="left"
).merge(
    dim_race,
    on="raceId",
    how="left"
)

In [39]:
sprint_points_team = (
    sprint_gold
    .groupby("team_name")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

**Exportar arquivos tratados**

In [42]:
GOLD_PATH.mkdir(parents=True, exist_ok=True)

f1_gold.to_csv(GOLD_PATH / "f1_resultados_gold.csv", index=False)
pit_gold.to_csv(GOLD_PATH / "f1_pit_stops_gold.csv", index=False)
lap_gold.to_csv(GOLD_PATH / "f1_lap_times_gold.csv", index=False)
qualifying_gold.to_csv(GOLD_PATH / "f1_qualifying_gold.csv", index=False)
sprint_gold.to_csv(GOLD_PATH / "f1_sprint_gold.csv", index=False)

In [44]:
import os

os.listdir()

['.config',
 'constructor_results.csv',
 'lap_times.csv',
 'constructors.csv',
 'f1_qualifying_gold.csv',
 'circuits.csv',
 'constructor_standings.csv',
 'f1_pit_stops_gold.csv',
 'f1_lap_times_gold.csv',
 'pit_stops.csv',
 'results.csv',
 'drivers.csv',
 'sprint_results.csv',
 'races.csv',
 'seasons.csv',
 'f1_sprint_gold.csv',
 'status.csv',
 'driver_standings.csv',
 'f1_resultados_gold.csv',
 'data',
 'qualifying.csv',
 'sample_data']

In [47]:
!zip f1_analytics_gold.zip \
f1_resultados_gold.csv \
f1_pit_stops_gold.csv \
f1_lap_times_gold.csv \
f1_qualifying_gold.csv \
f1_sprint_gold.csv

  adding: f1_resultados_gold.csv (deflated 89%)
  adding: f1_pit_stops_gold.csv (deflated 82%)
  adding: f1_lap_times_gold.csv (deflated 87%)
  adding: f1_qualifying_gold.csv (deflated 82%)
  adding: f1_sprint_gold.csv (deflated 82%)
